# Masking quality and training robustness

**Role:** Analysis.

Load completed random-fate-masking checkpoints and matched zero-mask controls. Compare intact prediction quality, exactly-one-neighbor-masked prediction quality, and stability across training seeds and masking rates. Each benchmark uses saved outer folds, fitted transforms and training/early-stopping memberships; no models are fitted here. Missing fate is distinguished from the observed unassigned identity. Export support, uncertainty diagnostics and intervention comparisons in the selected benchmark directory.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.


## Analysis settings

Choose the input run or cached report and all analysis options here before model loading, inference or plotting. Derived paths and model-dependent defaults are resolved in the following cells; each checkpoint restores its saved data and transformations.

In [ ]:
import numpy as np
import torch

# Input runs, data and destinations
TRAINING_RUN = None  # Saved run name in training_results/fate_masking_training.

# Sampling and plot selection
BENCHMARK_CENTERS_PER_IDENTITY = 2  # Per-identity recipient quota for effect benchmarks.
QUALITY_CENTERS_PER_ORGANOID = 16  # Fate-independent recipient quota for quality benchmarks.
BENCHMARK_SAMPLING_SEED = 1701  # Random seed for reproducible sampling or initialization.

# Inference resources
INFERENCE_BATCH_SIZE = 128  # Number of graphs/subgraphs evaluated per inference batch.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Use CUDA when available, otherwise CPU.

# Statistical support
BOOTSTRAP_SAMPLES = 500  # Number of organoid bootstrap resamples.
MIN_ORGANOIDS = 5  # Minimum distinct organoids required for reported effects.

# Execution and display
RUN_BENCHMARKS = False  # Generate saved intact and masked quality benchmarks.


## Masking figure definitions

These study-specific definitions build the summaries or panels displayed below. Cohort selection, marker labels, thresholds and figure layouts remain visible here; shared numerical and plotting primitives are imported from `src/`. Defining these functions does not run inference or load results.

In [ ]:
MASKING_METRICS = ['mse_mu','mse_z','bias_z','nll_z','coverage_50','coverage_80','coverage_95',
           'standardized_error','standardized_error2']  # Quantities included in the study summaries.

import pandas as pd
import matplotlib.pyplot as plt
from src.analysis.interventions.size_sweeps import summarize_by_organoid

MASKING_COLORS = ['#555555','#3579b8','#d56a24','#6a9b42','#9063a6']  # Display colors assigned to groups or model variants.

def masking_add_size_bins(frame):
    frame=frame.copy()
    frame['size_bin']=pd.cut(frame.observed_n,[0,150,250,350,500,800,np.inf],right=False,labels=False)
    return frame

def masking_benchmark_tables(result, *, bootstrap_samples=500):
    """Intervals average seeds within organoids; seed spread is exported separately."""
    intact=masking_add_size_bins(result['intact']); masked=masking_add_size_bins(result['masked']) if not result['masked'].empty else result['masked']
    pairs=masking_add_size_bins(result['pair_effects']) if not result['pair_effects'].empty else result['pair_effects']
    rows=[]
    for metric in MASKING_METRICS:
        tab=summarize_by_organoid(intact,['mask_rate','center_marker','size_bin'],metric,
                                 bootstrap_samples=bootstrap_samples)
        pos=intact.groupby(['mask_rate','center_marker','size_bin'],observed=True).observed_n.median().rename('n').reset_index()
        tab=tab.merge(pos,on=['mask_rate','center_marker','size_bin']);tab['metric']=metric;rows.append(tab)
    intact_summary=pd.concat(rows,ignore_index=True)
    seed_quality=intact[intact.center_marker=='All'].groupby(['mask_rate','seed'])[MASKING_METRICS].mean().reset_index()
    if masked.empty:
        return dict(intact=intact_summary,seed_quality=seed_quality,masked=pd.DataFrame(),
                    effects=pd.DataFrame(),seed_effects=pd.DataFrame(),pair_quality=pd.DataFrame())
    quality=[]
    for state in ['intact','mask']:
        for metric in MASKING_METRICS:
            tab=summarize_by_organoid(masked,['mask_rate','hop','size_bin'],f'{state}_{metric}',
                                      bootstrap_samples=bootstrap_samples)
            pos=masked.groupby(['mask_rate','hop','size_bin'],observed=True).observed_n.median().rename('n').reset_index()
            tab=tab.merge(pos,on=['mask_rate','hop','size_bin']);tab['metric']=metric;tab['state']=state;quality.append(tab)
    for metric in ['delta_mse_z','delta_mse_mu']:
        tab=summarize_by_organoid(masked,['mask_rate','hop','size_bin'],metric,bootstrap_samples=bootstrap_samples)
        pos=masked.groupby(['mask_rate','hop','size_bin'],observed=True).observed_n.median().rename('n').reset_index()
        tab=tab.merge(pos,on=['mask_rate','hop','size_bin']);tab['metric']=metric;tab['state']='difference';quality.append(tab)
    keys=['mask_rate','seed','center_marker','source_marker_name','hop']
    # Equal organoid weight, even when an organoid has multiple eligible centers.
    seed_effects=pairs.groupby(keys+['organoid_str'],observed=True).agg(
        effect=('mask_delta_relative','mean'),delta_mse_z=('delta_mse_z','mean')).reset_index()
    seed_effects=seed_effects.groupby(keys,observed=True).agg(effect=('effect','mean'),
        delta_mse_z=('delta_mse_z','mean'),n_organoids=('organoid_str','nunique')).reset_index()
    effects=pairs.groupby(keys+['size_bin','organoid_str'],observed=True).mask_delta_relative.mean().reset_index()
    effects=effects.groupby(keys+['size_bin'],observed=True).agg(
        effect=('mask_delta_relative','mean'),n_organoids=('organoid_str','nunique')).reset_index()
    pair_rows=[]
    for state in ['intact','mask']:
        for metric in ['mse_z','bias_z','coverage_95']:
            tab=summarize_by_organoid(pairs,['mask_rate','center_marker','source_marker_name','hop','size_bin'],
                                      f'{state}_{metric}',bootstrap_samples=bootstrap_samples)
            tab['state']=state;tab['metric']=metric;pair_rows.append(tab)
    return dict(intact=intact_summary,masked=pd.concat(quality,ignore_index=True),seed_quality=seed_quality,
                seed_effects=seed_effects,effects=effects,pair_quality=pd.concat(pair_rows,ignore_index=True))

def masking_line(ax, part, color, label, *, style='-'):
    part=part.sort_values('n')
    ax.plot(part.n,part['mean'],color=color,label=label,ls=style,marker='.')
    ax.fill_between(part.n,part.ci_low,part.ci_high,color=color,alpha=.10)
    ax.set_xscale('log');ax.set_xlabel('Observed N (bin median)')

def masking_plot_intact_quality(tables):
    frame=tables['intact'];frame=frame[frame.center_marker=='All']
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for rate,color in zip(sorted(frame.mask_rate.unique()),MASKING_COLORS):
        for ax,metric,title in zip(axes,['mse_mu','mse_z','coverage_95'],
                                  ['Curvature MSE','Transformed-target MSE','95% predictive coverage']):
            masking_line(ax,frame[(frame.mask_rate==rate)&(frame.metric==metric)],color,f'{rate:.0%}')
            ax.set_title(title)
    axes[-1].axhline(.95,color='black',ls=':',lw=1);axes[-1].set_ylim(0,1)
    axes[0].legend(title='Training mask probability')
    fig.suptitle('Intact outer-fold predictions: matched unmasked control and masking models')
    fig.tight_layout();return fig

def masking_plot_masked_quality(tables, *, hop=1):
    frame=tables['masked'];frame=frame[frame.hop==hop]
    fig,axes=plt.subplots(1,3,figsize=(16,4))
    for rate,color in zip(sorted(frame.mask_rate.unique()),MASKING_COLORS[1:]):
        for state,style in [('intact','-'),('mask','--')]:
            part=frame[(frame.mask_rate==rate)&(frame.metric=='mse_z')&(frame.state==state)]
            masking_line(axes[0],part,color,f'{rate:.0%} {state}',style=style)
        masking_line(axes[1],frame[(frame.mask_rate==rate)&(frame.metric=='delta_mse_z')],color,f'{rate:.0%}')
        masking_line(axes[2],frame[(frame.mask_rate==rate)&(frame.metric=='coverage_95')&(frame.state=='mask')],color,f'{rate:.0%}')
    axes[0].set_title('Same recipients: intact vs one neighbor hidden');axes[0].set_ylabel('MSE, transformed target')
    axes[1].set_title('Masking-induced change in prediction error');axes[1].axhline(0,color='.5',lw=1)
    axes[1].set_ylabel('Masked MSE − intact MSE')
    axes[2].set_title('Masked 95% predictive coverage');axes[2].axhline(.95,color='black',ls=':',lw=1);axes[2].set_ylim(0,1)
    axes[0].legend(fontsize=8)
    fig.suptitle(f'Exactly one source hidden; recipient identity observed • hop {hop}')
    fig.tight_layout();return fig

def masking_plot_training_robustness(result, tables):
    quality=tables['seed_quality'];masked=result['masked']
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for seed,frame in quality.groupby('seed'):
        frame=frame.sort_values('mask_rate')
        control=float(frame.loc[frame.mask_rate==0,'mse_z'].iloc[0])
        axes[0].plot(frame.mask_rate*100,frame.mse_z-control,marker='o',label=f'Seed {seed}')
    axes[0].axhline(0,color='.5',lw=1);axes[0].set_title('Intact MSE change vs matched 0% control')
    if not masked.empty:
        org=masked.groupby(['mask_rate','seed','organoid_str'])[['mask_mse_z','mask_nll_z']].mean().reset_index()
        means=org.groupby(['mask_rate','seed'])[['mask_mse_z','mask_nll_z']].mean().reset_index()
        for seed,frame in means.groupby('seed'):
            frame=frame.sort_values('mask_rate')
            axes[1].plot(frame.mask_rate*100,frame.mask_mse_z,marker='o',label=f'Seed {seed}')
            axes[2].plot(frame.mask_rate*100,frame.mask_nll_z,marker='o',label=f'Seed {seed}')
    axes[1].set_title('Single-neighbor masked MSE');axes[2].set_title('Single-neighbor masked Gaussian NLL')
    for ax in axes:ax.set_xlabel('Training masking probability (%)')
    axes[0].legend();fig.suptitle('Training robustness: one curve per seed, equal organoid weight')
    fig.tight_layout();return fig

def masking_plot_effect_robustness(result,tables,*,min_organoids=5):
    frame=tables['seed_effects'];names=result['config']['markers']+['Unassigned']
    fig,axes=plt.subplots(1,2,figsize=(15,6))
    for ax,hop in zip(axes,[1,2]):
        part=frame[(frame.hop==hop)&(frame.n_organoids>=min_organoids)].copy()
        part['positive']=(part.effect>0).astype(float)
        support=part.groupby(['center_marker','source_marker_name']).size()
        fraction=part.groupby(['center_marker','source_marker_name']).positive.mean()
        expected=len(result['config']['seeds'])*sum(p>0 for p in result['config']['training']['rates'])
        fraction=fraction.where(support==expected).unstack().reindex(index=names,columns=names)
        img=ax.imshow(fraction.to_numpy(),vmin=0,vmax=1,cmap='coolwarm')
        ax.set_xticks(range(len(names)),names,rotation=45,ha='right');ax.set_yticks(range(len(names)),names)
        ax.set_xlabel('Original source identity');ax.set_ylabel('Unchanged center identity');ax.set_title(f'Hop {hop}')
    fig.subplots_adjust(right=.87,bottom=.23,wspace=.45,top=.86)
    cax=fig.add_axes([.91,.25,.015,.5]);fig.colorbar(img,cax=cax,label='Fraction of rate/seed means > 0')
    fig.suptitle('Pooled observed-size masking effects: sign consistency across rates and seeds\n'
                 'Descriptive only: weak effects can change sign; magnitudes and N-bin profiles are exported.')
    return fig

## Load analysis inputs

Resolve the choices above and load the source measurements or saved model artifacts used by the following sections.

In [ ]:
import json
from IPython.display import display

assert (ROOT/'src').is_dir()
from src.analysis.interventions.masking import MaskTrainingConfig, load_mask_benchmarks
# Study-specific functions are defined in this notebook.

## Training settings

Every optimizer step averages an intact and randomly masked pass with equal loss weight. In the masked pass, each node is hidden independently with the same probability, including naturally unassigned cells. Masks change every batch and do not use fate, N, curvature, geometry, or crypt labels. Rates share the same random uniforms, data order and base initialization within a seed. The 0% control also performs two passes per step, so dropout, normalization passes and optimizer-step counts are comparable. Additional mask-input weights start at zero.

Each original outer training fold is split into fitting and **inner early-stopping** organoids. All variants select checkpoints by intact inner-validation transformed-target MSE. Original outer validation organoids are reserved for the benchmarks and ablations. Cached baseline/target preprocessing is reused from the original outer-fold run; it is not refitted on the inner fitting subset. Existing cohort-level outlier interpolation is retained. This is not a claim of fully nested preprocessing or causal unbiasedness.

The original model used a different training/selection protocol. The newly trained **0% control** is the direct benchmark for masking augmentation; historical checkpoint comparisons remain secondary.

In [ ]:
# Saved model selection
from src.artifacts.paths import resolve_training_run
MASKING_RUN = resolve_training_run(ROOT, 'fate_masking_training', TRAINING_RUN)
REFERENCE_RUN = MASKING_RUN / 'reference'  # Exact reference cohort/preprocessing copied with masking training.


# Benchmark sampling

# Inference resources

# Statistical summaries

# Inference resources
torch.set_num_threads(4)
print('Device:',DEVICE)
print('Saved training settings:', json.loads((MASKING_RUN/'settings.json').read_text())['training'])
print('Output:',MASKING_RUN)

In [ ]:
from src.artifacts.provenance import workflow_fingerprint
import copy
from dataclasses import asdict, dataclass
import hashlib
import pickle
import shutil
from scipy.stats import norm
from torch import nn
from torch_geometric.loader import DataLoader
from src.analysis.size_conditioning.cohort_inputs import make_model, seed_all, load_cohort, fold_graphs
from src.analysis.interventions.replacement import identity_codes, make_cases, _write_csv, UNASSIGNED
from src.analysis.interventions.size_sweeps import _size_override
from src.analysis.interventions.perturbation import predict_subgraph_center_distribution
from src.data.metadata import load_marker_names_from_dir
from src.data.subgraphs import build_ego_subgraphs_for_graph
from src.graph.neighborhood import compute_hop_rings
from src.inference.predict import predict_targets
from src.training.losses import CompositeLoss, make_base_loss, WeightedLossTerm, edge_loss_term
from src.analysis.interventions.masking import rate_tag, encode_fates, random_mask, ObservedFateAdapter, _forward_with_mask, make_mask_model, inner_split, _loss, intact_validation_mse, train_mask_model, checkpoint_path, load_mask_checkpoint, _sha, checked_settings, score_arrays, benchmark_intact, evaluate_single_mask, make_quality_cases
WORKFLOW_SOURCE = PROJECT_ROOT / 'experiments/benchmarks/masking_quality_and_robustness.ipynb'  # Notebook used when recording the code fingerprint.


### Benchmark masking experiment

The study sequence is defined here; shared numerical operations are imported above.

In [ ]:
from src.artifacts import pickle_compat as artifact_pickle
def benchmark_masking_experiment(root, output_dir, *, centers_per_identity=2, sampling_seed=1701,
                                 quality_centers_per_organoid=16, batch_size=128, device=None):
    """Score full intact validation graphs and exactly-one-neighbor masking cases."""
    root, out = Path(root).resolve(), Path(output_dir).resolve()
    if not (out / 'training_complete.json').exists():
        raise RuntimeError('Complete model training before the benchmark.')
    cfg = json.loads((out / 'settings.json').read_text()); settings=cfg['model_settings']; markers=cfg['markers']
    reference = out / 'reference'; device=device or ('cuda' if torch.cuda.is_available() else 'cpu')
    paths = [checkpoint_path(out,f,s,p) for f in cfg['folds'] for s in cfg['seeds'] for p in cfg['training']['rates']]
    checked_settings(out / 'benchmarks/settings.json', dict(centers_per_identity=centers_per_identity,
        quality_centers_per_organoid=quality_centers_per_organoid,
        sampling_seed=sampling_seed, batch_size=batch_size, code_sha256=workflow_fingerprint(WORKFLOW_SOURCE),
        training_settings_sha256=_sha(out/'settings.json'), checkpoints={p.name:_sha(p) for p in paths}))
    graphs=load_cohort(root/'training_data'/settings['DATASET_NAME'], settings, reference)
    refs=pd.read_csv(reference/'geometric_normalization/references.csv').set_index('fold')
    for split in json.loads((reference/'splits.json').read_text()):
        fold=split['fold']
        if fold not in cfg['folds']: continue
        with (reference/f'checkpoints/fold_{fold}_preprocessing.pkl').open('rb') as f: pre=artifact_pickle.load(f)
        val=fold_graphs(graphs,split,pre,settings)['val']
        subs,cases,_=make_cases(val,markers,centers_per_identity=centers_per_identity,
                                sweep_centers_per_identity=1,seed=sampling_seed+fold, receptive_hops=max(2, settings['NUM_LAYERS']))
        _write_csv(cases,out/'benchmarks'/f'fold_{fold}_cases.csv.gz')
        quality_subs,quality_cases=make_quality_cases(val,markers,
            centers_per_organoid=quality_centers_per_organoid,seed=sampling_seed+10000+fold,
            receptive_hops=max(2, settings['NUM_LAYERS']))
        _write_csv(quality_cases,out/'benchmarks'/f'fold_{fold}_quality_cases.csv.gz')
        for rate in cfg['training']['rates']:
            for seed in cfg['seeds']:
                stem=f'fold_{fold}_seed_{seed}_{rate_tag(rate)}'
                intact_path=out/'benchmarks'/f'{stem}_intact.csv.gz'
                mask_path=out/'benchmarks'/f'{stem}_masked.csv.gz'
                pair_path=out/'benchmarks'/f'{stem}_pair_effects.csv.gz'
                if intact_path.exists() and (rate==0 or (mask_path.exists() and pair_path.exists())): continue
                model=load_mask_checkpoint(out,fold,seed,rate,settings,markers)
                metadata=dict(fold=fold,seed=seed,mask_rate=rate)
                if not intact_path.exists():
                    intact=benchmark_intact(val,model,pre['residual_transform'],markers,batch_size=batch_size,device=device)
                    _write_csv(intact.assign(**metadata),intact_path)
                # An untrained missingness flag is not a valid masked baseline.
                if rate>0:
                    for selected_subs,selected_cases,path in [(quality_subs,quality_cases,mask_path),(subs,cases,pair_path)]:
                        if path.exists(): continue
                        frame=evaluate_single_mask(selected_subs,selected_cases,model,pre['residual_transform'],
                            size_center=pre['size_center'],size_scale=pre['size_scale'],batch_size=batch_size,device=device)
                        factor=np.exp(refs.loc[fold,'alpha']+refs.loc[fold,'beta']*np.log(frame.observed_n))/(4*np.pi)
                        frame['mask_delta_relative']=frame.mask_delta_mu*factor
                        frame['zero_delta_relative']=frame.zero_delta_mu*factor
                        _write_csv(frame.assign(**metadata),path)
                print(f'Benchmarked {stem}',flush=True)
    (out/'benchmarks/complete.json').write_text('{}')
    return out/'benchmarks'

## Benchmark on outer validation organoids

Intact benchmarking evaluates every cell in each full validation graph. For prediction quality and marginal calibration, recipient centers are sampled uniformly within organoid, then one source is sampled uniformly from each exact hop-1/hop-2 ring, without consulting identities. These cases are fixed across rates/seeds. Separately, identity-stratified cases cover all eight identities (including Unassigned) for pair-effect diagnostics. The center always stays observed. This separation prevents oversampling rare source identities from changing the hidden-fate prior in aggregate calibration scores.

The 0% control is evaluated on intact inputs only: its mask flag was never trained, so masking it would not define a valid unknown-fate reference. For each positive-rate model, zeroing is also evaluated as a within-model reference. There is no hypothetical-size accuracy score: only observed-size inputs have measured targets.

Saved uncertainty metrics use the predicted Gaussian in transformed-target space: NLL, standardized residual moments, and central 50%, 80%, 95% coverage. Coverage also applies after the monotone inverse transform; physical-space MSE uses inverse-transformed mean predictions. These metrics do not include model-parameter uncertainty.

In [ ]:
if RUN_BENCHMARKS:
    benchmark_masking_experiment(ROOT,MASKING_RUN,
        centers_per_identity=BENCHMARK_CENTERS_PER_IDENTITY,
        quality_centers_per_organoid=QUALITY_CENTERS_PER_ORGANOID,
        sampling_seed=BENCHMARK_SAMPLING_SEED,
        batch_size=INFERENCE_BATCH_SIZE,device=DEVICE)
bench=load_mask_benchmarks(MASKING_RUN)
tables=masking_benchmark_tables(bench,bootstrap_samples=BOOTSTRAP_SAMPLES)
for name,table in tables.items():
    table.to_csv(MASKING_RUN/'benchmarks'/f'summary_{name}.csv',index=False)
display(tables['seed_quality'])
if (MASKING_RUN/'validation_mse.csv').exists():
    baseline_comparison = pd.read_csv(MASKING_RUN/'validation_mse.csv')
    display(baseline_comparison.groupby('rate')[['mse','baseline_mse','mse_minus_baseline']].mean())


## 1. Intact prediction quality

MSE is computed within organoid, then organoids receive equal weight. Curvature errors are on the baseline-residual scale; the unchanged per-organoid baseline cancels from prediction errors. Shading is a pointwise organoid bootstrap after averaging repeated seeds. It conditions on fitted models; seed spread is shown separately.

All-center curves are displayed here. `tables['intact']` additionally contains each center identity and size bin, including bias, NLL, standardized errors, and 50/80/95% coverage.

In [ ]:
fig=masking_plot_intact_quality(tables)
fig.savefig(MASKING_RUN/'figures/01_intact_quality.png',dpi=160,bbox_inches='tight')
plt.show()

## 2. Exactly-one-neighbor masked prediction quality

Solid lines show intact predictions for the **same selected recipient/source cases** used by dashed masked lines. This avoids comparing the masked cohort against a different collection of cells. The error-change panel uses paired differences, averaging cases and seeds within organoid.

Detailed center/source/hop/N quality strata are saved in `summary_pair_quality.csv`; low-support strata should not be interpreted as reliable calibration estimates. Negative signed curvature contrasts and increases/decreases in prediction error are different quantities.

Masked errors stratified by the original hidden source identity can have nonzero mean even for a correctly calibrated conditional predictor: the predictor is not told the identity used to define that stratum. Use the fate-independent quality sample for marginal calibration; pair-stratified errors are descriptive diagnostics.

In [ ]:
for hop in (1,2):
    fig=masking_plot_masked_quality(tables,hop=hop)
    fig.savefig(MASKING_RUN/f'figures/02_masked_quality_hop{hop}.png',dpi=160,bbox_inches='tight')
    plt.show()
display(tables['pair_quality'].query('n_organoids >= @MIN_ORGANOIDS').head(16))

## 3. Training robustness

Each line below represents one model seed, aggregated across held-out organoids. Compare both intact performance relative to the newly trained 0% control and masked prediction quality across rates. The following identity matrix shows sign consistency of **pooled observed-size** masking effects across all trained positive rates and seeds. Values near 0 or 1 indicate agreement on direction; values near 0.5 indicate disagreement. This is descriptive, not a significance test; tiny effects can flip sign.

`summary_seed_effects.csv` retains effect magnitudes and support. `summary_effects.csv` also retains N-bin profiles for every rate/seed and center/source/hop combination, so pooled sign agreement need not be mistaken for agreement of curve shape.

In [ ]:
fig=masking_plot_training_robustness(bench,tables)
fig.savefig(MASKING_RUN/'figures/03_rate_seed_quality.png',dpi=160,bbox_inches='tight')
plt.show()
fig=masking_plot_effect_robustness(bench,tables,min_organoids=MIN_ORGANOIDS)
fig.savefig(MASKING_RUN/'figures/04_effect_sign_robustness.png',dpi=160,bbox_inches='tight')
plt.show()
display(tables['seed_effects'].head(16))

## Continue with static and swept masking ablation

Open **ablation/size_dependent_ablation.ipynb**, set `MASKING_RUN` to this directory, and choose a positive trained masking rate. That notebook reconstructs the exact saved replacement cases and reuses their replacement weights. It evaluates zeroing, both replacement mixtures and masking **within the same masking-trained checkpoint**, keeping historical model results separately labelled.

Masking curves retain the original replacement-supported cohort for the four-way comparison. This is a comparability restriction; missing-fate inference itself does not require donor matches. No rate is automatically selected from the outer validation benchmark—any selection after inspecting it is exploratory.